In [1]:
import pandas as pd
df = pd.read_parquet("hf://datasets/ChristophSchuhmann/MS_COCO_2017_URL_TEXT/mscoco.parquet")
df.head()

,URL,TEXT
0,http://images.cocodataset.org/train2017/000000...,A man with a red helmet on a small moped on a ...
1,http://images.cocodataset.org/train2017/000000...,Man riding a motor bike on a dirt road on the ...
2,http://images.cocodataset.org/train2017/000000...,A man riding on the back of a motorcycle.
3,http://images.cocodataset.org/train2017/000000...,A dirt path with a young person on a motor bik...
4,http://images.cocodataset.org/train2017/000000...,A man in a red shirt and a red hat is on a mot...


In [2]:
df

,URL,TEXT
0,http://images.cocodataset.org/train2017/000000...,A man with a red helmet on a small moped on a ...
1,http://images.cocodataset.org/train2017/000000...,Man riding a motor bike on a dirt road on the ...
2,http://images.cocodataset.org/train2017/000000...,A man riding on the back of a motorcycle.
3,http://images.cocodataset.org/train2017/000000...,A dirt path with a young person on a motor bik...
4,http://images.cocodataset.org/train2017/000000...,A man in a red shirt and a red hat is on a mot...
...,...,...
591748,http://images.cocodataset.org/train2017/000000...,The patrons enjoy their beverages at the bar.
591749,http://images.cocodataset.org/train2017/000000...,People having a drink in a basement bar.
591750,http://images.cocodataset.org/train2017/000000...,A group of friends enjoys a drink while sittin...
591751,http://images.cocodataset.org/train2017/000000...,Group of people drinking wine at a public loca...


In [3]:
df[['URL', 'TEXT']].nunique()

URL     118287
TEXT    570281
dtype: int64

In [4]:
from nltk.corpus import stopwords
import string
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize (split into words)
    tokens = text.split()
    # Remove stopwords and potentially short words (optional)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    return tokens



In [5]:
# apply the preprocessing function to the 'text' column
df['processed_text'] = df['TEXT'].apply(preprocess_text)
df

,URL,TEXT,processed_text
0,http://images.cocodataset.org/train2017/000000...,A man with a red helmet on a small moped on a ...,"[man, red, helmet, small, moped, dirt, road]"
1,http://images.cocodataset.org/train2017/000000...,Man riding a motor bike on a dirt road on the ...,"[man, riding, motor, bike, dirt, road, country..."
2,http://images.cocodataset.org/train2017/000000...,A man riding on the back of a motorcycle.,"[man, riding, back, motorcycle]"
3,http://images.cocodataset.org/train2017/000000...,A dirt path with a young person on a motor bik...,"[dirt, path, young, person, motor, bike, rests..."
4,http://images.cocodataset.org/train2017/000000...,A man in a red shirt and a red hat is on a mot...,"[man, red, shirt, red, hat, motorcycle, hill, ..."
...,...,...,...
591748,http://images.cocodataset.org/train2017/000000...,The patrons enjoy their beverages at the bar.,"[patrons, enjoy, beverages, bar]"
591749,http://images.cocodataset.org/train2017/000000...,People having a drink in a basement bar.,"[people, drink, basement, bar]"
591750,http://images.cocodataset.org/train2017/000000...,A group of friends enjoys a drink while sittin...,"[group, friends, enjoys, drink, sitting, bar]"
591751,http://images.cocodataset.org/train2017/000000...,Group of people drinking wine at a public loca...,"[group, people, drinking, wine, public, location]"


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json


corpus = df['processed_text'].apply(lambda x: ' '.join(x)).tolist()

# Initialize the TF-IDF vectorizer with your preferred preprocessing settings:
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)

def search_with_tfidf(query, top_n=20):
    # Vectorize the user query
    query_vec = vectorizer.transform([query])
    # Compute cosine similarities between the query and all image captions
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    # Get the indices of the top_n similar items
    top_indices = similarities.argsort()[::-1][:top_n]
    # Retrieve the corresponding image IDs and raw descriptions
    results = []
    for idx in top_indices:
        results.append({
            'image_id': df.iloc[idx]['URL'],
            'raw_description': df.iloc[idx]['TEXT'],
            'similarity_score': similarities[idx]
        })
    return results


# Example usage:
query = "A painting of a lone figure on a mountain under a starry sky"
results = search_with_tfidf(query)
for result in results:
    print(result['image_id'], result['raw_description'], result['similarity_score'])


http://images.cocodataset.org/train2017/000000289746.jpg a blurred photograph with starry decal of a young woman 0.39068501399601224
http://images.cocodataset.org/train2017/000000130706.jpg A  HORSE IS STANDING A LONE AMONG A MOUNTAIN 0.3882595321701432
http://images.cocodataset.org/train2017/000000349318.jpg a painting of an airplane in the sky  0.3809983433613583
http://images.cocodataset.org/train2017/000000032244.jpg a painting of a elephant standing with the sky behind him  0.36570546920991537
http://images.cocodataset.org/train2017/000000415943.jpg The front of a clock with two windows containing the figure of a woman in one and the figure of a man in the other. 0.35276460045777075
http://images.cocodataset.org/train2017/000000034193.jpg A painting of two bears with a mountain in the background. 0.34920778095311156
http://images.cocodataset.org/train2017/000000200830.jpg This is a painting of a boy painting. 0.3430730302284326
http://images.cocodataset.org/train2017/000000526663.

In [7]:
# most common tokens occuring together let's say 2 words
from sklearn.feature_extraction.text import CountVectorizer
# Initialize the CountVectorizer
count_vectorizer = CountVectorizer(ngram_range=(1, 10), stop_words='english')
# Fit the model to the corpus
count_vectorizer.fit(corpus)
# Get the vocabulary
vocab = count_vectorizer.vocabulary_
# Get the counts of each n-gram
X = count_vectorizer.transform(corpus)
# Sum the counts across all documents
sum_counts = X.sum(axis=0)
# Create a DataFrame with the n-grams and their counts
ngram_counts = pd.DataFrame(sum_counts.T, index=count_vectorizer.get_feature_names_out(), columns=['count'])
# Sort the DataFrame by count
ngram_counts = ngram_counts.sort_values(by='count', ascending=False)
# Display the most common n-grams
ngram_counts.head(10)

,count
man,73025
sitting,52905
standing,42473
people,41914
white,36178
woman,33951
street,30270
table,30172
holding,27967
large,24554


In [8]:
len(count_vectorizer.get_feature_names_out())


4261606

In [9]:
# len(ngram_counts) where count > 100
# get the most common n-grams
ngram_counts[ngram_counts['count'] > 50]

,count
man,73025
sitting,52905
standing,42473
people,41914
white,36178
...,...
standing store,51
melted cheese,51
licks,51
pizza meat,51


In [10]:
import numpy as np
from tqdm import tqdm

# Get the ngrams with counts > 50
frequent_ngrams = ngram_counts[ngram_counts['count'] > 50].index.tolist()

# Get the indices of these ngrams in the vocabulary
frequent_indices = [count_vectorizer.vocabulary_[ngram] for ngram in frequent_ngrams]

# Create a dataframe to store the results
result_data = []

# For each frequent ngram
for ngram, ngram_idx in tqdm(zip(frequent_ngrams, frequent_indices)):
    # Get the count from ngram_counts
    count = ngram_counts.loc[ngram]['count']
    
    # Find documents that contain this ngram
    # Get the column from the count matrix corresponding to this ngram
    ngram_col = X[:, ngram_idx].toarray().flatten()
    
    # Find indices where count > 0
    doc_indices = np.where(ngram_col > 0)[0]
    
    # Get the corresponding document IDs
    doc_ids = df.iloc[doc_indices]['URL'].unique().tolist()
    unique_count = len(doc_ids)
    
    # Add to results
    result_data.append({
        'ngram': ngram,
        'count': count,
        'document_ids': doc_ids,
        'unique_count': unique_count,
    })

# Create the final DataFrame
ngram_ids_df = pd.DataFrame(result_data)

# Sort by count in descending order
ngram_ids_df = ngram_ids_df.sort_values(by='count', ascending=False)

# Display the result
ngram_ids_df.head()

11385it [03:34, 53.00it/s]


,ngram,count,document_ids,unique_count
0,man,73025,[http://images.cocodataset.org/train2017/00000...,26596
1,sitting,52905,[http://images.cocodataset.org/train2017/00000...,31400
2,standing,42473,[http://images.cocodataset.org/train2017/00000...,25358
3,people,41914,[http://images.cocodataset.org/train2017/00000...,18397
4,white,36178,[http://images.cocodataset.org/train2017/00000...,23910


In [11]:
# first entry dict
ngram_ids_df.iloc[0].to_dict()

{'ngram': 'man',
 'count': 73025,
 'document_ids': ['http://images.cocodataset.org/train2017/000000391895.jpg',
  'http://images.cocodataset.org/train2017/000000184613.jpg',
  'http://images.cocodataset.org/train2017/000000483108.jpg',
  'http://images.cocodataset.org/train2017/000000293802.jpg',
  'http://images.cocodataset.org/train2017/000000113588.jpg',
  'http://images.cocodataset.org/train2017/000000384553.jpg',
  'http://images.cocodataset.org/train2017/000000079841.jpg',
  'http://images.cocodataset.org/train2017/000000412151.jpg',
  'http://images.cocodataset.org/train2017/000000229643.jpg',
  'http://images.cocodataset.org/train2017/000000387362.jpg',
  'http://images.cocodataset.org/train2017/000000001146.jpg',
  'http://images.cocodataset.org/train2017/000000097434.jpg',
  'http://images.cocodataset.org/train2017/000000463836.jpg',
  'http://images.cocodataset.org/train2017/000000122851.jpg',
  'http://images.cocodataset.org/train2017/000000085160.jpg',
  'http://images.coc

In [12]:
from collections import Counter
import re

# Function to extract the most common words from a set of descriptions
def get_common_words(document_ids, original_df, n=100):
    # Get all the descriptions for these document IDs
    descriptions = original_df[original_df['URL'].isin(document_ids)]['TEXT'].tolist()
    
    # Join all descriptions into one text
    all_text = ' '.join(descriptions)
    
    # Convert to lowercase and remove punctuation
    all_text = all_text.lower()
    all_text = re.sub(r'[^\w\s]', '', all_text)
    
    # Tokenize
    words = all_text.split()
    
    # Remove stopwords
    words = [word for word in words if word not in stop_words and len(word) > 2]
    
    # Count frequencies
    word_counts = Counter(words)
    
    # Get the n most common words
    common_words = [word for word, count in word_counts.most_common(n)]
    
    # Join back into a string
    return ' '.join(common_words)

# Add the labels column to the DataFrame
ngram_ids_df['labels'] = ngram_ids_df['document_ids'].apply(
    lambda doc_ids: get_common_words(doc_ids, df) + ' ' + ngram_ids_df.iloc[0]['ngram'] + ' ' + ngram_ids_df.iloc[0]['ngram'] + ' ' + ngram_ids_df.iloc[1]['ngram'] + ' ' + ngram_ids_df.iloc[1]['ngram'] + ' ' + ngram_ids_df.iloc[2]['ngram'] 
)

# Display the updated DataFrame
ngram_ids_df.head()

,ngram,count,document_ids,unique_count,labels
0,man,73025,[http://images.cocodataset.org/train2017/00000...,26596,man person holding standing riding tennis peop...
1,sitting,52905,[http://images.cocodataset.org/train2017/00000...,31400,sitting table next man two white top people ca...
2,standing,42473,[http://images.cocodataset.org/train2017/00000...,25358,standing man people two next woman field holdi...
3,people,41914,[http://images.cocodataset.org/train2017/00000...,18397,people group two man standing sitting street w...
4,white,36178,[http://images.cocodataset.org/train2017/00000...,23910,white sitting man black plate bathroom next tw...


In [13]:
ngram_ids_df.sort_values(by='unique_count', ascending=False).head(30)

,ngram,count,document_ids,unique_count,labels
1,sitting,52905,[http://images.cocodataset.org/train2017/00000...,31400,sitting table next man two white top people ca...
0,man,73025,[http://images.cocodataset.org/train2017/00000...,26596,man person holding standing riding tennis peop...
2,standing,42473,[http://images.cocodataset.org/train2017/00000...,25358,standing man people two next woman field holdi...
4,white,36178,[http://images.cocodataset.org/train2017/00000...,23910,white sitting man black plate bathroom next tw...
9,large,24554,[http://images.cocodataset.org/train2017/00000...,18480,large sitting people white standing clock two ...
3,people,41914,[http://images.cocodataset.org/train2017/00000...,18397,people group two man standing sitting street w...
15,near,19570,[http://images.cocodataset.org/train2017/00000...,16670,near standing two man next people sitting stre...
10,person,24412,[http://images.cocodataset.org/train2017/00000...,15722,person man holding riding woman standing sitti...
8,holding,27967,[http://images.cocodataset.org/train2017/00000...,15587,holding man woman tennis standing person peopl...
13,small,20088,[http://images.cocodataset.org/train2017/00000...,15154,small sitting white next two bathroom standing...


In [14]:
# count - unique document_ids and ngram
ngram_ids_df['unique_document_ids'] = ngram_ids_df['document_ids'].apply(lambda x: len(set(x)))
# difference between count and unique_document_ids
ngram_ids_df['count'] - ngram_ids_df['unique_document_ids']


0        46429
1        21505
2        17115
3        23517
4        12268
         ...  
11254        2
11255        0
11256       12
11257        6
11384        1
Length: 11385, dtype: int64

In [15]:
# keep unique document ids
ngram_ids_df['document_ids'] = ngram_ids_df['document_ids'].apply(lambda x: list(set(x)))

In [16]:
# First, we need to create a TF-IDF matrix for our collections (n-grams)
# We'll use the 'labels' column as our corpus

# Prepare the corpus from the labels column
collection_corpus = ngram_ids_df['labels'].tolist()

# Initialize a new TF-IDF vectorizer for collections
collection_vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
collection_tfidf_matrix = collection_vectorizer.fit_transform(collection_corpus)


In [17]:

def search_collections(query, top_n=10):
    """
    Search for collections based on a text query.
    
    Args:
        query (str): The search query
        top_n (int): Number of top results to return
        
    Returns:
        list: List of dictionaries containing collection information
    """
    # Vectorize the user query
    query_vec = collection_vectorizer.transform([query])
    
    # Compute cosine similarities between the query and all collections
    similarities = cosine_similarity(query_vec, collection_tfidf_matrix).flatten()
    
    # Get the indices of the top_n similar items
    top_indices = similarities.argsort()[::-1][:top_n]
    
    # Retrieve the corresponding collection information
    results = []
    for idx in top_indices:
        results.append({
            'ngram': ngram_ids_df.iloc[idx]['ngram'],
            'count': ngram_ids_df.iloc[idx]['count'],
            'similarity_score': similarities[idx],
            'labels': ngram_ids_df.iloc[idx]['labels'],
            'images': ngram_ids_df.iloc[idx]['document_ids'],
            'sample_images': ngram_ids_df.iloc[idx]['document_ids'][:5]  # First 5 images as examples
        })
    
    return results


In [18]:

# Example usage
collection_query = "butterflies fly away, butterflies in the sky"
collection_results = search_collections(collection_query)

# Display the results
for i, result in enumerate(collection_results):
    print(f"Result {i+1}:")
    print(f"N-gram: {result['ngram']}")
    print(f"Count: {result['count']}")
    print(f"Similarity score: {result['similarity_score']:.4f}")
    print(f"Labels (first 10 words): {' '.join(result['labels'].split()[:10])}...")
    print(f"Sample images: {result['sample_images']}")
    print("-" * 50)

Result 1:
N-gram: butterfly
Count: 66
Similarity score: 0.2825
Labels (first 10 words): kite butterfly flying man woman standing girl people sitting orange...
Sample images: ['http://images.cocodataset.org/train2017/000000238402.jpg', 'http://images.cocodataset.org/train2017/000000133315.jpg', 'http://images.cocodataset.org/train2017/000000565459.jpg', 'http://images.cocodataset.org/train2017/000000320200.jpg', 'http://images.cocodataset.org/train2017/000000392687.jpg']
--------------------------------------------------
Result 2:
N-gram: sitting orange
Count: 53
Similarity score: 0.2081
Labels (first 10 words): orange sitting next two top sign bowl plate red chair...
Sample images: ['http://images.cocodataset.org/train2017/000000563730.jpg', 'http://images.cocodataset.org/train2017/000000473612.jpg', 'http://images.cocodataset.org/train2017/000000496513.jpg', 'http://images.cocodataset.org/train2017/000000217103.jpg', 'http://images.cocodataset.org/train2017/000000099388.jpg']
--------

In [19]:
import pickle
import os
# Code to save the models and data (would be run separately, not part of the module)
def save_models(image_vectorizer, collection_vectorizer, image_tfidf_matrix, 
                collection_tfidf_matrix, df, ngram_ids_df, model_dir='saved_models'):
    """
    Save all the necessary models and data to disk
    
    Args:
        image_vectorizer: The TF-IDF vectorizer for images
        collection_vectorizer: The TF-IDF vectorizer for collections
        image_tfidf_matrix: The TF-IDF matrix for images
        collection_tfidf_matrix: The TF-IDF matrix for collections
        df: The original DataFrame with image data
        ngram_ids_df: The DataFrame with collection data
        model_dir: Directory to save the models and data
    """
    # Create the directory if it doesn't exist
    os.makedirs(model_dir, exist_ok=True)
    
    # Save the vectorizers
    with open(os.path.join(model_dir, 'image_vectorizer.pkl'), 'wb') as f:
        pickle.dump(image_vectorizer, f)
    
    with open(os.path.join(model_dir, 'collection_vectorizer.pkl'), 'wb') as f:
        pickle.dump(collection_vectorizer, f)
    
    # Save the TF-IDF matrices
    with open(os.path.join(model_dir, 'image_tfidf_matrix.pkl'), 'wb') as f:
        pickle.dump(image_tfidf_matrix, f)
    
    with open(os.path.join(model_dir, 'collection_tfidf_matrix.pkl'), 'wb') as f:
        pickle.dump(collection_tfidf_matrix, f)
    
    # Save the DataFrames
    df.to_pickle(os.path.join(model_dir, 'original_df.pkl'))
    ngram_ids_df.to_pickle(os.path.join(model_dir, 'ngram_ids_df.pkl'))

In [20]:
save_models(
    vectorizer, collection_vectorizer, tfidf_matrix, collection_tfidf_matrix, df, ngram_ids_df
)

In [21]:
# 'sitting basket' collection
# get the collection
collection_query = "sitting basket"
collection_results = search_collections(collection_query)
# Display the results
for i, result in enumerate(collection_results):
    print(f"Result {i+1}:")
    print(f"N-gram: {result['ngram']}")
    print(f"Count: {result['count']}")
    print(f"Similarity score: {result['similarity_score']:.4f}")
    print(f"Labels (first 10 words): {' '.join(result['labels'].split()[:10])}...")
    print(f"Sample images: {result['sample_images']}")
    print("-" * 50)

Result 1:
N-gram: sitting red
Count: 313
Similarity score: 0.2056
Labels (first 10 words): red sitting cat next man dog two table top chair...
Sample images: ['http://images.cocodataset.org/train2017/000000010256.jpg', 'http://images.cocodataset.org/train2017/000000309948.jpg', 'http://images.cocodataset.org/train2017/000000183757.jpg', 'http://images.cocodataset.org/train2017/000000056580.jpg', 'http://images.cocodataset.org/train2017/000000097779.jpg']
--------------------------------------------------
Result 2:
N-gram: bike
Count: 3826
Similarity score: 0.1952
Labels (first 10 words): bike man motorcycle riding bicycle street parked next person sitting...
Sample images: ['http://images.cocodataset.org/train2017/000000082338.jpg', 'http://images.cocodataset.org/train2017/000000426201.jpg', 'http://images.cocodataset.org/train2017/000000215563.jpg', 'http://images.cocodataset.org/train2017/000000353370.jpg', 'http://images.cocodataset.org/train2017/000000479095.jpg']
-----------------